# Exercise 2 — Results of the hyperparameter search

Multilayer perceptron for digit classification (0–9), trained on `digits.csv`.

This notebook only **reads** the runs that `src/perceptrons/sweep.py` saved in `results/`; it trains nothing.
`results/` is gitignored, so to reproduce the figures, run the sweeps first:

```bash
uv run python -m perceptrons.sweep
```

**Contents**

1. Setup and loading of all runs
2. How performance is evaluated
3. Learning rate
4. Architecture
5. Optimizer
6. Seed variation of the best candidates
7. Additional hyperparameters: activation function and batch size
8. Learning rate at batch size 8
9. Summary of the search path
10. Final configuration
11. Test set (production)
12. Open items

## 1. Setup

In [ ]:
import json
from math import ceil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS_DIR = Path("../results")
DATA_DIR = Path("../data")

# Categorical palette, used in this fixed order (never cycled)
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
INK, INK_SECONDARY, INK_MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": AXIS,
    "axes.labelcolor": INK_SECONDARY,
    "axes.titlecolor": INK,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "xtick.color": INK_MUTED,
    "ytick.color": INK_MUTED,
    "legend.frameon": False,
    "legend.fontsize": 9,
    "lines.linewidth": 2,
    "lines.markersize": 8,
})

In [ ]:
def load_runs(results_dir: Path) -> pd.DataFrame:
    # One row per saved run on digits.csv (more_digits runs belong to Exercise 3)
    rows = []
    for path in sorted(results_dir.glob("digits_*.json")):
        run = json.loads(path.read_text())
        config, result = run["config"], run["result"]
        valid_loss = result["valid_loss"]
        rows.append({
            "optimizer": config["optimizer_method"],
            "lr": config["learning_rate"],
            "hidden": str(config["hidden_layers"]),
            "batch_size": config["batch_size"],
            "activation": config["activation_method"],
            "beta": config["activation_parameter"],
            "epochs": config["epochs"],
            "seed": config["random_seed"],
            "valid_acc": 100 * result["accuracy"],
            "final_train_loss": result["train_loss"][-1],
            "final_valid_loss": valid_loss[-1],
            "best_valid_loss": min(valid_loss),
            "best_epoch": int(np.argmin(valid_loss)) + 1,
            "seconds": result["total_seconds"],
            "train_loss": result["train_loss"],
            "valid_loss": valid_loss,
            "recall": result["recall"],
            "cm": np.array(result["confusion_matrix"]),
        })
    return pd.DataFrame(rows)


def select(**conditions) -> pd.DataFrame:
    # Rows of `runs` matching every column == value pair
    mask = pd.Series(True, index=runs.index)
    for column, value in conditions.items():
        mask &= runs[column] == value
    return runs[mask]


TABLE_COLUMNS = ["optimizer", "lr", "hidden", "batch_size", "activation", "seed",
                 "valid_acc", "final_train_loss", "final_valid_loss", "best_valid_loss", "best_epoch", "seconds"]


def table(df: pd.DataFrame) -> pd.DataFrame:
    # Compact, rounded view for display
    return (df[TABLE_COLUMNS]
            .round({"valid_acc": 2, "final_train_loss": 4, "final_valid_loss": 4, "best_valid_loss": 4, "seconds": 1})
            .reset_index(drop=True))


def plot_loss_curves(ax, df: pd.DataFrame, label_column: str, show_train: bool = False, title: str = "") -> None:
    # Validation loss per epoch (solid); optionally training loss (dashed, same colour)
    for i, (_, row) in enumerate(df.iterrows()):
        epochs = np.arange(1, len(row["valid_loss"]) + 1)
        color = SERIES[i]
        ax.plot(epochs, row["valid_loss"], color=color, label=f"{label_column} = {row[label_column]}")
        if show_train:
            ax.plot(epochs, row["train_loss"], color=color, linestyle="--", linewidth=1.5, alpha=0.8)
    ax.set_yscale("log")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE loss (log scale)")
    ax.set_title(title)
    ax.legend()


def plot_heatmap(ax, matrix: np.ndarray, fmt: str, vmin: float, vmax: float, colorbar_label: str | None = None) -> None:
    # Matrix as coloured cells with the value written in each cell (dark text on light cells, white on dark)
    image = ax.imshow(matrix, cmap="Blues", vmin=vmin, vmax=vmax)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix[i, j]
            dark_cell = (value - vmin) / (vmax - vmin) > 0.5
            ax.text(j, i, format(value, fmt), ha="center", va="center", fontsize=7,
                    color="white" if dark_cell else INK)
    ax.set_xticks(range(matrix.shape[1]))
    ax.set_yticks(range(matrix.shape[0]))
    ax.grid(False)
    if colorbar_label is not None:
        ax.figure.colorbar(image, ax=ax, label=colorbar_label)


runs = load_runs(RESULTS_DIR)
print(f"{len(runs)} runs loaded")

## 2. How performance is evaluated

**Data split**

| Set | Source | Use |
|---|---|---|
| Training | first 80 % of `digits.csv` | learning the weights |
| Validation | last 20 % of `digits.csv` | choosing hyperparameters (all comparisons in this notebook) |
| Test | `digits_test.csv` | evaluated **once**, on the final configuration only (section 11) |

- The file is not sorted by label, so the unshuffled 80/20 split is not biased towards any class (checked in `ex3_data_exploration.ipynb`).
- No exact duplicates within the files and no overlap between `digits.csv` and `digits_test.csv` (same notebook).

**Metrics**

- **Accuracy** on the validation set: the selection criterion.
- **MSE loss** per epoch (training and validation): shows convergence and overfitting.
- **Per-class recall** from the confusion matrix: shows which digits are weak.

**Caveats that affect every number below**

- `digits.csv` has **no samples of class 8**, so validation accuracy is accuracy over 9 classes.
- Accuracy is measured after the **last** epoch (50), not at the best epoch.
- One validation sample ≈ 0.04 percentage points (computed below), so small differences are only a handful of images.

In [ ]:
n_total = 12449
n_train = int(0.8 * n_total)
n_valid = n_total - n_train
print(f"training samples:   {n_train}")
print(f"validation samples: {n_valid}  ->  one image = {100 / n_valid:.3f} percentage points")

# Upper bound on test accuracy for any model trained on digits.csv (it never sees an 8)
test_labels = pd.read_csv(DATA_DIR / "digits_test.csv", usecols=["label"])["label"]
n_test, n_test_8 = len(test_labels), int((test_labels == 8).sum())
print(f"\ntest samples: {n_test}, of which class 8: {n_test_8} ({100 * n_test_8 / n_test:.1f} %)")
print(f"max. possible test accuracy: {100 * (n_test - n_test_8) / n_test:.1f} %")

## 3. Learning rate

Gradient descent, architecture `[16]`, batch size 32, sigmoid (β = 1), 50 epochs, seed 0.
The values are on a logarithmic scale, so each step multiplies the learning rate by ~3.

In [ ]:
lr_runs = select(optimizer="gradient_descent", hidden="[16]", batch_size=32, activation="sigmoid", seed=0).sort_values("lr")
display(table(lr_runs))

fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(12, 4))
ax_acc.plot(lr_runs["lr"], lr_runs["valid_acc"], marker="o", color=SERIES[0])
ax_acc.set_xscale("log")
ax_acc.set_xlabel("Learning rate (log scale)")
ax_acc.set_ylabel("Validation accuracy (%)")
ax_acc.set_title("Validation accuracy after 50 epochs")
best = lr_runs.loc[lr_runs["valid_acc"].idxmax()]
ax_acc.annotate(f"{best['valid_acc']:.2f} %", (best["lr"], best["valid_acc"]),
                textcoords="offset points", xytext=(0, 8), ha="center", color=INK)

plot_loss_curves(ax_loss, lr_runs, "lr", title="Validation loss per epoch")
plt.tight_layout()
plt.show()

**Observations**

- Accuracy rises from 0.01 to 0.3 and drops slightly at 1.0 → **0.3** is chosen.
- 0.03 and 0.1 are still improving at epoch 50, so the ranking is "given 50 epochs".
- At 1.0 the validation loss stops improving after ~20 epochs while the training loss keeps falling.

## 4. Architecture

Gradient descent, learning rate 0.3, batch size 32, sigmoid, 50 epochs, seed 0.
Width is varied (`[16]`, `[64]`, `[128]`) and depth (`[64]` vs `[64, 32]`).

In [ ]:
architectures = ["[16]", "[64]", "[128]", "[64, 32]"]
arch_runs = select(optimizer="gradient_descent", lr=0.3, batch_size=32, activation="sigmoid", seed=0)
arch_runs = arch_runs[arch_runs["hidden"].isin(architectures)]
arch_runs = arch_runs.set_index("hidden").loc[architectures].reset_index()
display(table(arch_runs))

fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(12, 4))
ax_acc.plot(arch_runs["valid_acc"], arch_runs["hidden"], "o", color=SERIES[0])
for _, row in arch_runs.iterrows():
    ax_acc.annotate(f"{row['valid_acc']:.2f} %", (row["valid_acc"], row["hidden"]),
                    textcoords="offset points", xytext=(10, -3), color=INK)
ax_acc.invert_yaxis()
ax_acc.margins(x=0.15)
ax_acc.set_xlabel("Validation accuracy (%)")
ax_acc.set_ylabel("Hidden layers")
ax_acc.set_title("Validation accuracy after 50 epochs")

plot_loss_curves(ax_loss, arch_runs, "hidden", show_train=True,
                 title="Validation (solid) and training (dashed) loss")
plt.tight_layout()
plt.show()

**Observations**

- Width helps a lot at first (`[16]` → `[64]`: +1.6 pp), then little (`[64]` → `[128]`: +0.1 pp).
- The extra layer (`[64, 32]`) gives the lowest training loss but not a higher validation accuracy: the gap between training and validation loss is the largest.
- `[64]` and `[128]` are both carried forward to the seed comparison (section 6).

## 5. Optimizer

Architecture `[64]`, batch size 32, sigmoid, 50 epochs, seed 0.
Each optimizer gets **its own learning-rate sweep**: a good learning rate differs per optimizer, so comparing them at one shared learning rate would compare learning rates, not optimizers.

In [ ]:
optimizers = ["gradient_descent", "momentum", "adam"]
opt_runs = select(hidden="[64]", batch_size=32, activation="sigmoid", seed=0)
opt_runs = opt_runs[opt_runs["optimizer"].isin(optimizers)].sort_values(["optimizer", "lr"])
display(table(opt_runs))

fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(12, 4))
best_per_optimizer = []
for i, name in enumerate(optimizers):
    rows = opt_runs[opt_runs["optimizer"] == name]
    ax_acc.plot(rows["lr"], rows["valid_acc"], marker="o", color=SERIES[i], label=name)
    best_per_optimizer.append(rows.loc[rows["valid_acc"].idxmax()])
ax_acc.set_xscale("log")
ax_acc.set_xlabel("Learning rate (log scale)")
ax_acc.set_ylabel("Validation accuracy (%)")
ax_acc.set_title("Learning-rate sweep per optimizer")
ax_acc.legend()

best_per_optimizer = pd.DataFrame(best_per_optimizer)
plot_loss_curves(ax_loss, best_per_optimizer, "optimizer", show_train=True,
                 title="Best learning rate per optimizer: validation (solid) / training (dashed)")
plt.tight_layout()
plt.show()

display(table(best_per_optimizer))

**Observations**

- Momentum collapses at high learning rates (0.1 → 71.8 %, 0.3 → 37.6 %). Its effective step is roughly `lr / (1 − β_momentum)`, about 10× the learning rate, so the sigmoid neurons saturate (derivative ≈ 0) and learning stops.
- Momentum's best value (0.01) lies inside the tested range; it ends ~1 pp below the other two.
- Adam reaches a low training loss much faster, but its validation loss bottoms out around epoch 20–40 and then rises: it overfits within 50 epochs.
- Gradient descent and Adam end within ~0.2 pp of each other; section 6 checks whether that difference survives other seeds.

## 6. Seed variation of the best candidates

The seed sets the initial weights and the shuffling order. Each candidate is run with seeds 0, 1 and 2; the spread shows how large a difference has to be before it means anything.

In [ ]:
candidates = {
    "GD · lr 0.3 · [64] · bs 32": dict(optimizer="gradient_descent", lr=0.3, hidden="[64]", batch_size=32),
    "Adam · lr 0.001 · [64] · bs 32": dict(optimizer="adam", lr=0.001, hidden="[64]", batch_size=32),
    "GD · lr 0.3 · [128] · bs 32": dict(optimizer="gradient_descent", lr=0.3, hidden="[128]", batch_size=32),
    "GD · lr 0.3 · [128] · bs 8": dict(optimizer="gradient_descent", lr=0.3, hidden="[128]", batch_size=8),
    "GD · lr 1.0 · [128] · bs 8": dict(optimizer="gradient_descent", lr=1.0, hidden="[128]", batch_size=8),
}

summary_rows = []
for label, conditions in candidates.items():
    rows = select(activation="sigmoid", **conditions).sort_values("seed")
    acc = rows.set_index("seed")["valid_acc"]
    summary_rows.append({
        "candidate": label,
        "n_seeds": len(acc),
        "seed 0": acc.get(0), "seed 1": acc.get(1), "seed 2": acc.get(2),
        "mean": acc.mean(),
        "std": acc.std(ddof=1),   # sample standard deviation over the seeds
    })
seed_summary = pd.DataFrame(summary_rows).round(2)
display(seed_summary)

fig, ax = plt.subplots(figsize=(10, 3.8))
for i, row in seed_summary.iterrows():
    seeds = [row["seed 0"], row["seed 1"], row["seed 2"]]
    ax.plot(seeds, [i] * 3, "o", color=SERIES[0], alpha=0.55, label="single seed" if i == 0 else None)
    ax.plot(row["mean"], i, "D", color=INK, markersize=7, label="mean" if i == 0 else None)
    ax.annotate(f"{row['mean']:.2f} ± {row['std']:.2f}", (max(seeds), i),
                textcoords="offset points", xytext=(12, -3), color=INK)
ax.set_yticks(range(len(seed_summary)))
ax.set_yticklabels(seed_summary["candidate"])
ax.invert_yaxis()
ax.set_xlabel("Validation accuracy (%)")
ax.set_title("Validation accuracy per seed (dots) and mean (diamond)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2)
ax.margins(x=0.12)
plt.tight_layout()
plt.show()

**Observations**

- With batch size 32: `[128]` beats `[64]` on **every** seed (same seed = same starting point), and has the highest mean. Adam and GD `[64]` are indistinguishable.
- Seed spread is 0.1–0.3 pp, i.e. 2–7 validation images. Differences smaller than that are not evidence.
- The two batch-size-8 candidates come from sections 7 and 8; they are included here so all seed results sit in one table.

## 7. Additional hyperparameters

Not required by the assignment ("es razonable que decidan variar ... otros hiperparámetros").
Starting point: gradient descent, lr 0.3, `[128]`, batch size 32, sigmoid, seed 0.

### 7.1 Activation function: sigmoid vs. tanh

In [ ]:
act_runs = pd.concat([
    select(activation="sigmoid", optimizer="gradient_descent", lr=0.3, hidden="[128]", batch_size=32, seed=0),
    select(activation="tangent", optimizer="gradient_descent", hidden="[128]", batch_size=32, seed=0).sort_values("lr"),
])
display(table(act_runs))

act_runs = act_runs.assign(run=act_runs["activation"] + ", lr " + act_runs["lr"].astype(str))
fig, ax = plt.subplots(figsize=(8, 4))
plot_loss_curves(ax, act_runs, "run", title="Validation loss per epoch: sigmoid vs. tanh")
plt.tight_layout()
plt.show()

**Observations**

- Tanh was first run at lr 0.3 (chosen for sigmoid). Its derivative peaks at β = 1.0 versus β/2 = 0.5 for the sigmoid used here (`activations.py`), so the same learning rate gives tanh roughly twice the step size. Tanh therefore got its own learning rates.
- Even at its best (lr 0.03) tanh stays below sigmoid, and its validation loss jumps between epochs while the training loss falls smoothly.
- Hypothesis (not tested): tanh is also used in the output layer, whose outputs can be negative while the targets are one-hot 0/1.
- 0.03 is the lowest learning rate tried for tanh, so its optimum may lie lower.

### 7.2 Batch size

In [ ]:
bs_runs = select(optimizer="gradient_descent", lr=0.3, hidden="[128]", activation="sigmoid", seed=0)
bs_runs = bs_runs[bs_runs["batch_size"].isin([8, 32, 128])].sort_values("batch_size")
bs_runs = bs_runs.assign(updates_per_epoch=[ceil(n_train / bs) for bs in bs_runs["batch_size"]])
display(table(bs_runs).assign(updates_per_epoch=bs_runs["updates_per_epoch"].values))

fig, (ax_epoch, ax_updates) = plt.subplots(1, 2, figsize=(12, 4))
plot_loss_curves(ax_epoch, bs_runs, "batch_size", title="Validation loss per epoch")
for i, (_, row) in enumerate(bs_runs.iterrows()):
    updates = np.arange(1, len(row["valid_loss"]) + 1) * row["updates_per_epoch"]
    ax_updates.plot(updates, row["valid_loss"], color=SERIES[i], label=f"batch_size = {row['batch_size']}")
ax_updates.set_xscale("log")
ax_updates.set_yscale("log")
ax_updates.set_xlabel("Weight updates (log scale)")
ax_updates.set_ylabel("MSE loss (log scale)")
ax_updates.set_title("Same curves, per weight update")
ax_updates.legend()
plt.tight_layout()
plt.show()

**Observations**

- Batch size 8 gives the highest accuracy, 128 the lowest.
- With a fixed number of epochs a smaller batch makes more weight updates (≈1 245 per epoch at 8 vs. ≈78 at 128). The right plot shows that per *update* the curves lie much closer together: batch size 128 is not worse so much as unfinished after 50 epochs.
- Online learning (batch size 1) was **not run** because of its run time.

## 8. Learning rate at batch size 8

The learning rate 0.3 was chosen at batch size 32. Since the best learning rate depends on the batch size, it is re-tuned for batch size 8 (`[128]`, sigmoid, seed 0).

In [ ]:
bs8_runs = select(optimizer="gradient_descent", hidden="[128]", batch_size=8, activation="sigmoid", seed=0).sort_values("lr")
display(table(bs8_runs))

fig, (ax_acc, ax_cm) = plt.subplots(1, 2, figsize=(12, 4.5), gridspec_kw={"width_ratios": [1.2, 1]})
ax_acc.plot(bs8_runs["lr"], bs8_runs["valid_acc"], marker="o", color=SERIES[0])
ax_acc.set_xscale("log")
ax_acc.set_xlabel("Learning rate (log scale)")
ax_acc.set_ylabel("Validation accuracy (%)")
ax_acc.set_title("Validation accuracy at batch size 8")

dead = bs8_runs[bs8_runs["lr"] == 3.0].iloc[0]
plot_heatmap(ax_cm, dead["cm"], fmt="d", vmin=0, vmax=dead["cm"].max())
ax_cm.set_xlabel("Predicted class")
ax_cm.set_ylabel("Actual class")
ax_cm.set_title("lr 3.0: confusion matrix (validation)")
ax_cm.grid(False)
plt.tight_layout()
plt.show()

**Observations**

- Accuracy keeps rising up to lr 1.0, then the network **dies** at 3.0: training and validation loss are exactly 0.5000 for all 50 epochs, and every image is predicted as class 8 (accuracy 0 %, because the validation set has no 8s).
- Why exactly 0.5: all sigmoid outputs are saturated at ≈ 0, so per sample the loss is ½ · (0 − 1)² = ½ for the true class and 0 for the other nine. The derivative is ≈ 0, so no gradient flows back and the weights never recover.
- The optimum lies between 0.3 and 3.0; lr 1.0 is confirmed over 3 seeds in section 6 (97.26 ± 0.12 % vs. 96.73 ± 0.27 % for lr 0.3, better on every seed).

## 9. Summary of the search path

Each step changes one thing and keeps the best value found so far.

In [ ]:
steps = [
    ("start", "GD · lr 0.1 · [16] · bs 32", dict(optimizer="gradient_descent", lr=0.1, hidden="[16]", batch_size=32)),
    ("learning rate", "GD · lr 0.3 · [16] · bs 32", dict(optimizer="gradient_descent", lr=0.3, hidden="[16]", batch_size=32)),
    ("architecture", "GD · lr 0.3 · [128] · bs 32", dict(optimizer="gradient_descent", lr=0.3, hidden="[128]", batch_size=32)),
    ("optimizer", "GD kept (Adam equal, Momentum worse)", dict(optimizer="gradient_descent", lr=0.3, hidden="[128]", batch_size=32)),
    ("batch size", "GD · lr 0.3 · [128] · bs 8", dict(optimizer="gradient_descent", lr=0.3, hidden="[128]", batch_size=8)),
    ("lr at bs 8", "GD · lr 1.0 · [128] · bs 8", dict(optimizer="gradient_descent", lr=1.0, hidden="[128]", batch_size=8)),
]
path_rows = []
for step, label, conditions in steps:
    rows = select(activation="sigmoid", **conditions)
    path_rows.append({
        "step": step,
        "configuration": label,
        "seed 0": rows.loc[rows["seed"] == 0, "valid_acc"].iloc[0],
        "mean over seeds": rows["valid_acc"].mean(),
        "n_seeds": len(rows),
    })
search_path = pd.DataFrame(path_rows).round(2)
display(search_path)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(range(len(search_path)), search_path["seed 0"], marker="o", color=SERIES[0])
for i, row in search_path.iterrows():
    ax.annotate(f"{row['seed 0']:.2f}", (i, row["seed 0"]), textcoords="offset points", xytext=(0, 8), ha="center", color=INK)
ax.set_xticks(range(len(search_path)))
ax.set_xticklabels(search_path["step"])
ax.set_ylabel("Validation accuracy, seed 0 (%)")
ax.set_title("Validation accuracy after each step of the search")
plt.tight_layout()
plt.show()

**Limitation of a step-by-step search:** it assumes the hyperparameters are roughly independent. They are not (the learning rate depended on the optimizer, the activation and the batch size), which is why the learning rate was re-tuned after each of those changes.

## 10. Final configuration

In [ ]:
final_conditions = dict(optimizer="gradient_descent", lr=1.0, hidden="[128]", batch_size=8, activation="sigmoid")
final_runs = select(**final_conditions).sort_values("seed")

final_config = pd.DataFrame({
    "value": {
        "optimizer": "gradient descent",
        "learning rate": 1.0,
        "architecture": "784 → 128 → 10",
        "activation": "sigmoid (all layers), β = 1.0",
        "loss": "MSE",
        "batch size": 8,
        "epochs": 50,
        "dataset": "digits.csv (80 % train / 20 % validation)",
        "validation accuracy": f"{final_runs['valid_acc'].mean():.2f} ± {final_runs['valid_acc'].std(ddof=1):.2f} % (seeds 0, 1, 2)",
    }
})
display(final_config)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_loss_curves(ax, final_runs, "seed", show_train=True,
                 title="Final configuration: validation (solid) / training (dashed) loss, 3 seeds")
plt.tight_layout()
plt.show()

In [ ]:
final_seed0 = final_runs[final_runs["seed"] == 0].iloc[0]
cm = final_seed0["cm"]
row_sums = cm.sum(axis=1, keepdims=True)
cm_normalized = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums > 0)

fig, (ax_cm, ax_recall) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1.3, 1]})
plot_heatmap(ax_cm, cm_normalized, fmt=".2f", vmin=0, vmax=1, colorbar_label="Fraction of actual class")
ax_cm.set_xlabel("Predicted class")
ax_cm.set_ylabel("Actual class")
ax_cm.set_title("Confusion matrix, validation, seed 0 (row-normalized)\nclass 8 row is empty: no 8s in digits.csv")
ax_cm.grid(False)

recall = [np.nan if r is None else 100 * r for r in final_seed0["recall"]]
classes = np.arange(len(recall))
ax_recall.barh(classes, recall, color=SERIES[0], height=0.6)
for c, r in zip(classes, recall):
    ax_recall.annotate("n/a" if np.isnan(r) else f"{r:.1f}", (0 if np.isnan(r) else r, c),
                       textcoords="offset points", xytext=(4, -3), color=INK, fontsize=9)
ax_recall.set_yticks(classes)
ax_recall.invert_yaxis()
ax_recall.set_xlim(0, 105)
ax_recall.grid(axis="y", visible=False)
ax_recall.set_xlabel("Recall (%)")
ax_recall.set_ylabel("Class")
ax_recall.set_title("Per-class recall, validation, seed 0")
plt.tight_layout()
plt.show()

**Observations**

- All three seeds converge the same way; the training loss approaches 0 while the validation loss flattens, so the model fits the training data almost perfectly.
- Class **5** has the lowest recall (only 271 training samples in `digits.csv`, about 2 % of the data). Class 8 cannot be evaluated.

## 11. Test set (production)

The final configuration, evaluated **once** on `digits_test.csv` with `src/perceptrons/evaluate_test.py`, using the weights saved for seeds 0, 1 and 2.
Nothing below was used to choose the configuration.

In [ ]:
test_rows = []
for path in sorted(RESULTS_DIR.glob("test_digits_*.json")):
    run = json.loads(path.read_text())
    seed, test = run["config"]["random_seed"], run["test"]
    cm_test = np.array(test["confusion_matrix"])
    known = np.arange(10) != 8   # the 9 classes the model was trained on
    test_rows.append({
        "seed": seed,
        "valid_acc": final_runs.loc[final_runs["seed"] == seed, "valid_acc"].iloc[0],
        "test_acc": 100 * test["accuracy"],
        "test_acc_without_8": 100 * np.trace(cm_test) / cm_test[known].sum(),
        "recall_5": 100 * test["recall"][5],
        "recall_8": 100 * test["recall"][8],
        "cm": cm_test,
    })
test_runs = pd.DataFrame(test_rows).sort_values("seed").reset_index(drop=True)

metrics = ["valid_acc", "test_acc", "test_acc_without_8", "recall_5", "recall_8"]
test_summary = test_runs.set_index("seed")[metrics]
test_summary.loc["mean"] = test_runs[metrics].mean()
test_summary.loc["std"] = test_runs[metrics].std(ddof=1)
display(test_summary.round(2))
print(f"upper bound for a model that never saw an 8: {100 * (n_test - n_test_8) / n_test:.1f} %")

In [ ]:
upper_bound = 100 * (n_test - n_test_8) / n_test
groups = {
    "Validation (digits.csv, no 8s)": test_runs["valid_acc"],
    "Test, classes 0–7 and 9": test_runs["test_acc_without_8"],
    "Test, all 10 classes": test_runs["test_acc"],
}

fig, ax = plt.subplots(figsize=(9, 3.4))
for i, (label, values) in enumerate(groups.items()):
    ax.plot(values, [i] * len(values), "o", color=SERIES[0], alpha=0.55, label="single seed" if i == 0 else None)
    ax.plot(values.mean(), i, "D", color=INK, markersize=7, label="mean" if i == 0 else None)
    ax.annotate(f"{values.mean():.2f} ± {values.std(ddof=1):.2f} %", (values.max(), i),
                textcoords="offset points", xytext=(12, -3), color=INK)
ax.axvline(upper_bound, color=INK_MUTED, linestyle="--", linewidth=1.5)
ax.annotate(f"max. possible on test ({upper_bound:.1f} %)", (upper_bound, 2.35),
            textcoords="offset points", xytext=(6, 0), color=INK_SECONDARY, fontsize=9)
ax.set_yticks(range(len(groups)))
ax.set_yticklabels(groups.keys())
ax.invert_yaxis()
ax.margins(x=0.12, y=0.2)
ax.set_xlabel("Accuracy (%)")
ax.set_title("Final configuration: validation vs. test (3 seeds)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
cm_test = test_runs.loc[test_runs["seed"] == 0, "cm"].iloc[0]
cm_test_normalized = cm_test / cm_test.sum(axis=1, keepdims=True)

fig, (ax_cm, ax_eights) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1.3, 1]})
plot_heatmap(ax_cm, cm_test_normalized, fmt=".2f", vmin=0, vmax=1, colorbar_label="Fraction of actual class")
ax_cm.set_xlabel("Predicted class")
ax_cm.set_ylabel("Actual class")
ax_cm.set_title("Confusion matrix, test, seed 0 (row-normalized)")

# Where do the 8s go? One group of bars per predicted class, one bar per seed
classes = np.arange(10)
width = 0.27
for i, (_, row) in enumerate(test_runs.iterrows()):
    ax_eights.bar(classes + (i - 1) * width, row["cm"][8], width=width, color=SERIES[i], label=f"seed {row['seed']}")
ax_eights.set_xticks(classes)
ax_eights.set_xlabel("Predicted class")
ax_eights.set_ylabel(f"Number of test 8s (of {n_test_8})")
ax_eights.set_title("What the model predicts for the 8s in the test set")
ax_eights.grid(axis="x", visible=False)
ax_eights.legend()
plt.tight_layout()
plt.show()

**Observations**

- Test accuracy is 86.61 ± 0.27 %, against 97.26 ± 0.12 % on validation.
- Almost the entire gap is class 8: all 243 test 8s are misclassified (≈ 9.7 pp). Without class 8 the test accuracy is ≈ 96 %, close to validation.
- The model never predicts 8. The test 8s are mostly predicted as **3**, then 2, 9 and 5.
- Class 5 has the lowest recall among the known classes on test as well (83–88 %).
- The spread over seeds (± 0.27 pp) is of the same size as on validation.
- With `digits.csv` alone the 98 % target of Exercise 3 is out of reach, whatever the hyperparameters (upper bound 90.3 %).

## 12. Open items

- [x] **Test evaluation**: done in section 11.
- [ ] **Online learning** (batch size 1): not run; either run it or state the reason in the report.
- [ ] **Hand-off to Exercise 3**: save the final configuration as a file (e.g. `configs/ex2_best.json`).